# General-rank longitudinal collection

Collect **300 new NA Ranked Solo/Duo players: 50 each from Bronze, Silver, Gold, Platinum, Emerald and Diamond**. Iron, Master+, unranked players and the saved Master cohort are excluded. The existing Master run, notebooks and caches remain separate and read-only. No LoLalytics collection or modeling is included.

**Initial review is offline.** The single switch below remembers its saved setting: `None` uses the checkpoint (initially disabled), `True` enables collection, and `False` explicitly pauses it. After enabling once, reopening this notebook and **Run All** resumes the saved run, even if the edited switch was not saved. Collection stops while the laptop is off; it resumes only when this notebook is run again.

Before enabling, **stop the Master collector and every other process using this Riot key**. The repository-wide lock prevents competing collectors that use this helper. The old Master notebook does not take this lock, so it cannot be stopped automatically.

## 1. Imports and configuration

The study configuration is copied from the existing Master checkpoint: patch 16.19, queue 420, the original target/season boundaries and **170 initial games**. Only the player goal and sampling seed differ. History subsequently expands with earlier completed target games; it is not a rolling last-170 window. No 150-game or OTP eligibility minimum is imposed.

In [1]:
from pathlib import Path
from dataclasses import replace, asdict
import json,os,sys
import pandas as pd
from dotenv import load_dotenv

start=Path.cwd().resolve()
PROJECT_ROOT=next((p for p in [start,*start.parents]
    if (p/'hlr_model/longitudinal_pipeline.py').is_file()),None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Run from the repository root or hlr_model/.')
if str(PROJECT_ROOT) not in sys.path:sys.path.insert(0,str(PROJECT_ROOT))
from hlr_model.longitudinal_pipeline import Config
from hlr_model.broad_rank_pipeline import BroadConfig,BroadCollector

RUN_COLLECTION = True  # True: enable once; None: remember saved setting; False: pause.
SAMPLING_SEED = 161926
PAGES_PER_DIVISION = 3
OUTPUT_RUN = PROJECT_ROOT/'hlr_model/data/runs/broad_na_16_19_seed_161926'
MASTER_CHECKPOINT = PROJECT_ROOT/'hlr_model/data/runs/patch_16_19_seed_1619/metadata/checkpoint.json'
with MASTER_CHECKPOINT.open() as handle:
    original = json.load(handle)['identity']['config']
study = replace(Config(**original),player_count=300,random_seed=SAMPLING_SEED,allow_unverified_masters=False)
config = BroadConfig(seed=SAMPLING_SEED,league_pages_per_division=PAGES_PER_DIVISION,study=study)
assert study.historical_depth==170, 'Inspect original study settings before changing history depth.'
display(pd.Series({key:getattr(study,key) for key in ['player_count','queue','target_patch','historical_depth',
    'season_start','target_search_start','target_search_end']},name='Frozen study settings').to_frame())

,Frozen study settings
player_count,300
queue,420
target_patch,16.19
historical_depth,170
season_start,1767902400
target_search_start,1790164800
target_search_end,1791442800


## 2. Load environment, caches and checkpoint

`RIOT_API_KEY` is reloaded from the **repository-root `.env` with `override=True`** on every Run All. The key is sent only in an HTTPS header and is never saved in checkpoints, logs or outputs. On 401/403 the run saves progress and stops: replace the key in `.env` and Run All again with the same configuration and output folder.

The new run owns `metadata/checkpoint.sqlite` and `raw/matches/`. SQLite transactions retain configuration/schema, candidate order, selected/reserve state, exact-query response caches, discovery cursors, per-player states, failures and rate/retry information. Successful query responses are committed before cursor advancement; successful match responses are atomically saved before player advancement. Interrupted `.tmp` files are not valid cached matches. Completed statuses are reconciled with actual payload files on reopening.

Existing raw caches are referenced in place, including `hlr_model/data/runs/patch_16_19_seed_1619/raw/matches/`, `data/otp_raw_matches/`, and `lpm_model/data/otp_raw_matches/`. No legacy file is written. Shared uncached match IDs are downloaded once. Incompatible settings are rejected rather than reset or merged.

In [2]:
load_dotenv(PROJECT_ROOT/'.env',override=True)
if 'collector' in globals():collector.close()
collector = BroadCollector(PROJECT_ROOT,OUTPUT_RUN,config,enabled=RUN_COLLECTION,
    api_key=os.getenv('RIOT_API_KEY'))
print('Run folder:',OUTPUT_RUN.relative_to(PROJECT_ROOT))
print('Network collection enabled:',collector.state['enabled'])
print('Existing Master players excluded:',len(collector.state['excluded_master_ids']))
print('Checkpoint:',(OUTPUT_RUN/'metadata/checkpoint.sqlite').relative_to(PROJECT_ROOT))

Run folder: hlr_model/data/runs/broad_na_16_19_seed_161926
Network collection enabled: True
Existing Master players excluded: 200
Checkpoint: hlr_model/data/runs/broad_na_16_19_seed_161926/metadata/checkpoint.sqlite


## 3. Current progress and sampling frame

Discovery requests a bounded frame: pages **1–3 per division I–IV in each allowed tier** (or fewer when an empty page ends that stratum). Before any player is attempted, page responses are saved and the candidate order is frozen. Within division, candidates are ordered by a seeded hash of their PUUID; divisions are interleaved in a seeded order. All frame candidates beyond the selected slots are same-tier reserves. At selection, `/entries/by-puuid` verifies the current Solo/Duo tier/division. A player who has moved outside the frozen tier is definitively ineligible for that quota.

This gives 50 completed players per tier, **not population-proportional rank sampling or a uniform sample of all ranked NA players**. API pages and subsequent patch-activity eligibility can introduce selection bias. Frame and rank-check timestamps are retained. Current rank is a **collection-time label**, not verified rank during patch 16.19.

Only definitive exclusions (for example, verified rank ineligibility or fully discovered absence of valid target games) advance to the next frozen reserve. A failed/unfinished download stays pending or incomplete for the same player. No selection uses wins, OTP status or favorable modeled results. A bounded frame can exhaust before a quota is met; the collector stops explicitly rather than silently declaring success or changing the frame.

Official documentation checked during implementation on October 26, 2026: [League-v4](https://developer.riotgames.com/apis#league-v4/GET_getLeagueEntries) and [public schema fragment](https://developer.riotgames.com/api-details/league-v4) specify `/lol/league/v4/entries/{queue}/{tier}/{division}` with `page` and `LeagueEntryDTO` fields `puuid`, `queueType`, `tier`, `rank`; `/entries/by-puuid/{encryptedPUUID}` verifies all queues. [Match-v5](https://developer.riotgames.com/apis#match-v5/GET_getMatchIdsByPUUID) uses regional routing and second-based `startTime`/`endTime`, offset `start`, and `count` up to 100. Schema mismatches stop with work pending instead of guessing identities.

In [3]:
display(collector.progress())
print('Candidate order frozen:',bool(collector.state['candidate_order']))
collector.show()

,tier,goal,attempted,excluded,pending,incomplete,completed,reserves
0,BRONZE,50,0,0,0,0,0,0
1,SILVER,50,0,0,0,0,0,0
2,GOLD,50,0,0,0,0,0,0
3,PLATINUM,50,0,0,0,0,0,0
4,EMERALD,50,0,0,0,0,0,0
5,DIAMOND,50,0,0,0,0,0,0


Candidate order frozen: False
Completed eligible players: 0 / 300; stage=not_started; remaining=300
BRONZE 0/50 | SILVER 0/50 | GOLD 0/50 | PLATINUM 0/50 | EMERALD 0/50 | DIAMOND 0/50
HTTP/cache session: {}; raw cache: {}; network: {}


## 4. Run or resume collection

Enable once with `RUN_COLLECTION=True`, then Run All. Later leave it `True` or return to `None` to use the saved enabled state. Set it to `False` and Run All to persist a pause. **Keep `OUTPUT_RUN` and all study settings unchanged.**

Each selected player needs: an allowed verified Solo/Duo rank; at least one actual gameVersion-verified patch-16.19 target; finished target pagination; finished history retrieval; and every required payload processed. Initial history contains at most 170 season-bounded queue-420 games completed strictly before the earliest target's kickoff. Later features may add earlier targets only after completion. Genuine short histories are preserved when bounded queries finish successfully; failed queries are never called genuinely short.

Connection errors, timeouts, unavailable responses and transient server errors remain retryable with persisted capped exponential backoff and jitter until interrupted. The client honors advertised application/method/service windows and Retry-After, including persisted cooldowns. A fresh process additionally waits at least **121 seconds before its first uncached request** to protect restarts/shared-key history. Repeated outages can take a long time; interrupt to pause safely. An authentication failure stops immediately with a refresh-key message. See [Riot's rate-limit documentation](https://developer.riotgames.com/docs/portal#web-apis_rate-limiting).

Use the notebook interrupt to pause. Do not run another collector with this key. When all six quotas and required retrievals are complete, further Run All executions make no collection requests.

In [4]:
collector.collect()
display(collector.progress())

Rate/startup cooldown: 121s; progress is saved.
Rate/startup cooldown: 10s; progress is saved.
Completed eligible players: 1 / 300; stage=initial history; remaining=299
BRONZE 1/50 | SILVER 0/50 | GOLD 0/50 | PLATINUM 0/50 | EMERALD 0/50 | DIAMOND 0/50
HTTP/cache session: {'new_http_requests': 128}; raw cache: {'new_raw_matches': 53, 'raw_cache_hits': 106}; network: {'at': 1793033095.045114, 'status': 'ok'}
Completed eligible players: 1 / 300; stage=target discovery; remaining=299
BRONZE 1/50 | SILVER 0/50 | GOLD 0/50 | PLATINUM 0/50 | EMERALD 0/50 | DIAMOND 0/50
HTTP/cache session: {'new_http_requests': 130}; raw cache: {'new_raw_matches': 53, 'raw_cache_hits': 106}; network: {'at': 1793033095.710995, 'status': 'ok'}
Completed eligible players: 1 / 300; stage=target discovery; remaining=299
BRONZE 1/50 | SILVER 0/50 | GOLD 0/50 | PLATINUM 0/50 | EMERALD 0/50 | DIAMOND 0/50
HTTP/cache session: {'new_http_requests': 132}; raw cache: {'new_raw_matches': 53, 'raw_cache_hits': 106}; networ

Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.12/site-packages/IPython/core/interactiveshell.py", line 3577, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/var/folders/r9/mhc5twwx0blgr6rw0wr28jmc0000gn/T/ipykernel_50834/262183270.py", line 1, in <module>
    collector.collect()
  File "/Users/minhho-hoang/Documents/riot_proj/riot-game-prediction/hlr_model/broad_rank_pipeline.py", line 441, in collect
    if current['status'] not in ('completed','excluded'):self.process(current)
                                                         ^^^^^^^^^^^^^^^^^^^^^
  File "/Users/minhho-hoang/Documents/riot_proj/riot-game-prediction/hlr_model/broad_rank_pipeline.py", line 407, in process
    self.histories(p)
  File "/Users/minhho-hoang/Documents/riot_proj/riot-game-prediction/hlr_model/broad_rank_pipeline.py", line 364, in histories
    q,page,count=self.query(p,'history')
                 ^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/minhho-hoang/Do

## 5. Build and export existing-format longitudinal datasets

Export runs **only when all 300 eligible players and all retrievals are complete**. It reuses the project's `player_record`, `expanding_features`, and `quality_report`; no model or external matchup scrape is run. It writes `player_cohort.parquet`, `historical_matches.parquet`, `target_matches.parquet`, `longitudinal_modeling.parquet`, `player_coverage.parquet` and `collection_metadata.json` in a timestamped `processed/` snapshot. Cohort ID `broad_na_16_19` distinguishes it from Master data. Rank labels, rank-check times and completeness fields accompany the original-format features.

Previous exports remain intact. A completed export is reused on later Run All executions with unchanged data; an interrupted export without a completion marker is not accepted. Only played champion choices receive observed outcomes.

In [ ]:
if collector.done:
    output_path = collector.export()
    print('Complete output snapshot:',output_path.relative_to(PROJECT_ROOT))
else:
    print('No completed export: remaining eligible players =',config.study.player_count-len(collector.completed()))
    print('Pending/incomplete players stay in the same cohort; completed source data have not been overwritten.')

No completed export: remaining eligible players = 300
Pending/incomplete players stay in the same cohort; completed source data have not been overwritten.


## 6. Final coverage and completeness summary

Completion below means verified allowed current rank plus completed target/history discovery and available processed payloads—not attempted players. Attempted, definitively excluded, pending, incomplete, completed and reserves are reported separately.

**Offline validation:** `python -m pytest hlr_model/tests/test_broad_rank_pipeline.py -q` uses fixtures and mocked HTTP only. It covers allowed ranks/quotas, preserved query responses across crashes, interrupted pagination, shared/raw cache reuse, retryable failures versus exclusion, persisted cooldowns and authentication stops, configuration rejection, locking, corrupted-cache reconciliation, and strict completion-based expanding chronology. Notebook implementation/review does not start real collection.

**Limits:** actual Riot availability, retention and late indexing can limit patch coverage. API exhaustion describes returned recorded history, not lifetime play. Frozen frame pages can span different collection times, rank can change after selection, and inferred role is not verified pregame intent. A long outage remains pending, not complete. The laptop performs no collection while shut down; reopen this notebook and Run All to resume. An expired key is replaced in root `.env`, with no new cohort or reset.

In [ ]:
display(collector.progress())
collector.show()
if collector.done:
    print('COMPLETE: all six rank quotas and required payloads verified. Exports are available above.')
else:
    print('NOT COMPLETE: collection is paused, not started, or still has required work.')
print('Collection continues only while this notebook is running; reopening requires Run All.')

,tier,goal,attempted,excluded,pending,incomplete,completed,reserves
0,BRONZE,50,0,0,0,0,0,0
1,SILVER,50,0,0,0,0,0,0
2,GOLD,50,0,0,0,0,0,0
3,PLATINUM,50,0,0,0,0,0,0
4,EMERALD,50,0,0,0,0,0,0
5,DIAMOND,50,0,0,0,0,0,0


Completed eligible players: 0 / 300; stage=not_started; remaining=300
BRONZE 0/50 | SILVER 0/50 | GOLD 0/50 | PLATINUM 0/50 | EMERALD 0/50 | DIAMOND 0/50
HTTP/cache session: {}; raw cache: {}; network: {}
NOT COMPLETE: collection is paused, not started, or still has required work.
Collection continues only while this notebook is running; reopening requires Run All.
